# PZ data challenge: task set 4

**Task set 4: as task set 3, with unrecognised blends.** Objects within the Rubin PSF were merged into single objects by summing their fluxes, and a blend's training `redshift` is that of its brightest member. The files have no blend flag. Labels and evaluation are as in task set 3:
- spectroscopically selected objects use `redshift`;
- COSMOS-only objects use `redshift_manyband`, so their metrics include the label noise.

This notebook runs all four configurations of the task set (cardinal and flagship, 1yr and 10yr). For each one it:

1. trains the CNN ensemble on the training file, holding out `VAL_FRAC` of it for evaluation (test redshifts are hidden);
2. scores the held-out objects with the challenge's metrics. The point metrics (bias, scatter, |Δz| > 0.2 rate) match RAIL's biweight statistics, and the PIT metrics come from `qp.metrics.pit.PIT`, as in the challenge evaluator;
3. writes the submission files under their official names into `OUT_DIR`, shared by all four task-set notebooks:
   - the test-set p(z) (subtask 1), checked with the challenge's own format check;
   - the trained ensemble as a single `.pkl` (subtask 2), via `cnnpz.save_ensemble_file`.

The p(z) is **not yet calibrated**: each Gaussian's width is the spread of the ensemble members, so PIT scores are poor.

Run this from the repo root with the `qp` conda env as the kernel. The filter curves are the ones shipped with RAIL, which `pz_data_challenge` installs.

## Configuration

In [ ]:
from rail.utils.path_utils import find_rail_file

import cnnpz  # run from the repo root so this picks up the local source

TASKSET = 4
SIMS = ["cardinal", "flagship"]
SCENARIOS = ["1yr", "10yr"]

DATA_DIR = "pz_data_challenge/tests/public"
OUT_DIR = "outputs/znn"  # submission files for all task sets; the contents of the submission tarball

# Filter curves shipped with RAIL; the challenge's catalogs.yaml ("cardinal_roman_rubin") maps the
# catalogue columns to these DC2LSST_* and roman_* filters
FILTER_DIR = find_rail_file("examples_data/estimation_data/data/FILTER")

# band -> (catalogue magnitude column, filter curve file), ordered by wavelength
BANDS = {
    "u": ("mag_u_lsst", f"{FILTER_DIR}/DC2LSST_u.res"),
    "g": ("mag_g_lsst", f"{FILTER_DIR}/DC2LSST_g.res"),
    "r": ("mag_r_lsst", f"{FILTER_DIR}/DC2LSST_r.res"),
    "i": ("mag_i_lsst", f"{FILTER_DIR}/DC2LSST_i.res"),
    "z": ("mag_z_lsst", f"{FILTER_DIR}/DC2LSST_z.res"),
    "y": ("mag_y_lsst", f"{FILTER_DIR}/DC2LSST_y.res"),
    "Y": ("mag_Y_roman", f"{FILTER_DIR}/roman_Y106.res"),
    "J": ("mag_J_roman", f"{FILTER_DIR}/roman_J129.res"),
    "H": ("mag_H_roman", f"{FILTER_DIR}/roman_H158.res"),
}
REF_BAND = "i"  # magnitudes are normalized by this band
FEATURE_CONFIG = cnnpz.DEFAULT_FEATURE_CONFIG

VAL_FRAC = 0.2  # fraction of the training set held out for evaluation
N_SPLITS = 5  # ensemble members (K-fold); 10 in the full setup
EPOCHS = 100  # max epochs per member (early stopping usually ends sooner)
SEED = 42

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qp.metrics.pit import PIT
from sklearn.model_selection import train_test_split

from pz_data_challenge import scoring, submit_utils

os.makedirs(OUT_DIR, exist_ok=True)
print("cnnpz from", cnnpz.__file__)

## Reference redshifts and evaluation subsets

In [ ]:
SPEC_SURVEYS = ["DEEP2_LSST", "DESI_BGS", "DESI_ELG_LOP", "DESI_LRG", "VVDSf02", "zCOSMOS"]


def reference_redshifts(train):
    """Reference redshift per training object, and the held-out subsets to score."""
    is_spec = train[[c for c in SPEC_SURVEYS if c in train]].astype(bool).any(axis=1).to_numpy()
    is_cosmos = train["COSMOS"].astype(bool).to_numpy()
    is_cosmos_only = is_cosmos & ~is_spec
    z_spec = train["redshift"].to_numpy()
    z_manyband = train["redshift_manyband"].to_numpy()
    assert np.isnan(z_spec[is_cosmos_only]).all() and np.isfinite(z_spec[is_spec]).all()

    z_ref = np.where(is_cosmos_only, z_manyband, z_spec)
    assert np.isfinite(z_ref).all(), "every training object should have a reference redshift"

    # How noisy are the many-band labels? Measured on objects with both a spec-z and a many-band photo-z,
    # which are bright, so this understates the noise at the faint end.
    overlap = is_cosmos & is_spec
    dz = (z_manyband[overlap] - z_spec[overlap]) / (1 + z_spec[overlap])
    bias, _, scatter, _, outliers = cnnpz.get_biweight_mean_sigma_outlier(dz)
    print(
        f"spec-selected: {is_spec.mean():.1%}, COSMOS-only: {is_cosmos_only.mean():.1%}; many-band vs spec-z on "
        f"{overlap.sum()} overlap objects: bias {bias:.4f}, scatter {scatter:.4f}, |dz|>0.2 rate {outliers:.4f}"
    )

    subsets = {
        "held-out spec-selected (vs spec-z)": is_spec,
        "held-out COSMOS-only (vs many-band photo-z)": is_cosmos_only,
        "held-out all": np.ones(len(train), dtype=bool),
    }
    return z_ref, subsets

## Evaluation helpers

In [ ]:
redshift_bins = np.linspace(0, 2.5, 11)
imag_bins = np.linspace(18, 25.5, 11)


def evaluate(trained_models, X_eval, z_eval, mag_i, ids, title):
    """Point and PIT metrics of the ensemble on one set of objects."""
    pz = cnnpz.ensemble_predict(trained_models, X_eval, ids=ids)
    z_point = np.squeeze(pz.ancil["zmode"])

    point_stats, redshift_stats, imag_stats = cnnpz.get_all_stats(
        z_eval, z_point, mag_i, save=False, redshift_bins=redshift_bins, imag_bins=imag_bins
    )
    cnnpz.plot_stats(point_stats, redshift_stats, imag_stats, z_eval, z_point, redshift_bins, imag_bins, mag_i)
    plt.suptitle(f"{title} ({len(z_eval)} objects)", y=1.02)
    plt.show()

    mean, _, std, _, abs_outlier_rate = point_stats
    pit = PIT(pz, z_eval)
    values = {
        "mean": mean,
        "std": std,
        "abs_outlier_rate": abs_outlier_rate,
        "ks": pit.evaluate_PIT_KS().statistic,
        "CvM": pit.evaluate_PIT_CvM().statistic,
        "ksamp": pit.evaluate_PIT_anderson_ksamp().statistic,
        # qp derives this from a spline fit to the PIT quantiles; it breaks down (huge values) when PIT
        # piles up at 0 or 1. The challenge evaluator uses the same call.
        "outlier": pit.evaluate_PIT_outlier_rate(),
        "pit_outlier_direct": np.mean((pit.pit_samps < 0.0001) | (pit.pit_samps > 0.9999)),
    }
    return values


def tier_points(values):
    """Points against the challenge scoring tiers: one per nested range a metric falls inside."""
    return {name: sum(lo <= float(values[name]) <= hi for lo, hi in ranges) for name, ranges in scoring.metric_dict.items()}

## Run all configurations

For each configuration: train on the fit split, score the held-out subsets, then write the test-set p(z) and the model file. The model used for the submission files is the one trained on the fit split.

In [ ]:
results = {}

for SIM in SIMS:
    for SCENARIO in SCENARIOS:
        config = f"{SIM}_{SCENARIO}"
        print(f"\n========== task set {TASKSET}: {config} ==========")

        train_file = os.path.join(DATA_DIR, f"pz_challenge_taskset_{TASKSET}_{SIM}_training_{SCENARIO}.hdf5")
        test_file = os.path.join(DATA_DIR, f"pz_challenge_taskset_{TASKSET}_{SIM}_test_{SCENARIO}.hdf5")
        train = cnnpz.read_catalog(train_file)
        test = cnnpz.read_catalog(test_file)
        z_ref, subsets = reference_redshifts(train)

        # how representative is the training set of the test set?
        bins = np.linspace(16, 26, 41)
        plt.figure(figsize=[6, 3.5])
        plt.hist(test["mag_i_lsst"], bins, density=True, histtype="step", color="k", label="test")
        plt.hist(train["mag_i_lsst"], bins, density=True, histtype="step", label="training")
        plt.xlabel("mag_i_lsst")
        plt.title(config)
        plt.legend()
        plt.show()

        X, _ = cnnpz.catalog_to_XY(train, BANDS, REF_BAND, FEATURE_CONFIG)
        X_test, _ = cnnpz.catalog_to_XY(test, BANDS, REF_BAND, FEATURE_CONFIG)

        idx_fit, idx_val = train_test_split(np.arange(len(train)), test_size=VAL_FRAC, random_state=SEED)
        trained_models, histories = cnnpz.train_ensembles(
            cnnpz.build_model, X[idx_fit], z_ref[idx_fit], N_SPLITS=N_SPLITS, EPOCHS=EPOCHS, random_state=SEED
        )
        cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.1))

        val = train.iloc[idx_val]
        for name, mask in subsets.items():
            m = mask[idx_val]
            results[(config, name)] = evaluate(
                trained_models,
                X[idx_val][m],
                z_ref[idx_val][m],
                val["mag_i_lsst"].to_numpy()[m],
                val["object_id"].to_numpy()[m],
                title=f"{config}: {name}",
            )

        # submission files: subtask 1 p(z) and subtask 2 model
        estimate_file = os.path.join(OUT_DIR, f"pz_challenge_taskset_{TASKSET}_{SIM}_pz_estimate_{SCENARIO}.hdf5")
        model_file = os.path.join(OUT_DIR, f"pz_challenge_taskset_{TASKSET}_{SIM}_pz_model_{SCENARIO}.pkl")

        pz_test = cnnpz.ensemble_predict(trained_models, X_test, ids=test["object_id"].to_numpy())
        pz_test.write_to(estimate_file)
        cnnpz.save_ensemble_file(model_file, trained_models, BANDS, REF_BAND, FEATURE_CONFIG)

        checks = submit_utils.check_pz_submission_file(estimate_file, test_file)
        print(estimate_file, "checks passed:", checks)
        assert checks == [1, 2, 3, 4, 5, 6, 7], "submission file failed the challenge format checks"

## Summary

Metric values and points against the challenge scoring tiers (`pz_data_challenge.scoring.metric_dict`: three nested ranges per metric, one point per range), per configuration and held-out subset.

In [ ]:
table = pd.DataFrame(results).T
table.index.names = ["configuration", "subset"]

points = pd.DataFrame({key: tier_points(values) for key, values in results.items()}).T
points["total"] = points.sum(axis=1)
points.index.names = ["configuration", "subset"]
n_max = sum(len(r) for r in scoring.metric_dict.values())

print("metric values")
print(table.round(4).to_string())
print(f"\npoints (out of 3 per metric, {n_max} in total)")
print(points.to_string())